# XGBoost 再学習ノートブック v4

| セル | 内容 |
|------|------|
| セル1 | Google Drive マウント + セットアップ |
| セル1b | **GitHub→Drive データマージ（history.db最新化）** |
| セル2 | src/ 強制アップデート（GitHub最新） |
| セル3 | 展開予測モデル再学習（19特徴量ペースモデル） |
| セル4 | speed_index キャッシュ再構築 + 学習データ生成 |
| セル5 | XGB再学習（通常モデル） + キャリブレーション |
| セル6 | 残差学習モデル（市場コピー排除実験） |
| セル7 | 統合テスト（cal_prob合計・重要度確認） |
| セル8 | 再学習モデルを GitHub main にプッシュ（本番反映） |

### データフロー
```
GitHub Actions（毎週） → GitHub history.db（週次蓄積）
                              ↓ セル1b（マージ）
Drive history.db（マスター） → セル3〜5（再学習）→ セル8（GitHub push）
```

> 使い方: セル1→1b→2〜7を順に実行。結果に満足したらセル8で本番反映。
> 前提: Colabシークレットに GITHUB_PAT を登録し、「ノートブックからのアクセス」をON。

In [ ]:
# == セル1: セットアップ ===================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
# == セル1b: GitHub→Drive データマージ（history.db の最新化）================
# GitHub Actions が毎週蓄積した新レースを Drive の history.db に取り込む。
# Drive = 学習データのマスター（過去一括取得分を含む完全版）。
import subprocess, sqlite3, shutil, os

REPO_URL = 'https://github.com/hanagenuku/keiba_ai.git'
TMP_REPO = '/tmp/keiba_gh'
DRIVE_DB = f'{BASE_DIR}/data/history.db'

# Drive側の現状
_drv = sqlite3.connect(DRIVE_DB)
_drv_races = _drv.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
_drv_horses = _drv.execute("SELECT COUNT(*) FROM horse_history").fetchone()[0]
_drv_max = _drv.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
_drv.close()
print(f'Drive: {_drv_races}レース / {_drv_horses}出走 / 最新{_drv_max}')

# GitHubからshallow clone（history.dbだけ取得）
if os.path.exists(TMP_REPO):
    shutil.rmtree(TMP_REPO)

try:
    from google.colab import userdata
    _pat = userdata.get('GITHUB_PAT')
    _clone_url = f'https://{_pat}@github.com/hanagenuku/keiba_ai.git'
except Exception:
    _clone_url = REPO_URL

subprocess.run(
    f'git clone --depth 1 --filter=blob:none --sparse "{_clone_url}" {TMP_REPO}',
    shell=True, capture_output=True)
subprocess.run(
    f'git -C {TMP_REPO} sparse-checkout set data/history.db',
    shell=True, capture_output=True)
subprocess.run(
    f'git -C {TMP_REPO} checkout',
    shell=True, capture_output=True)

GH_DB = f'{TMP_REPO}/data/history.db'
if not os.path.exists(GH_DB) or os.path.getsize(GH_DB) < 1000:
    print('!! GitHub DB取得失敗（LFSポインタの可能性）。スキップします。')
    print('   Drive の history.db をそのまま使用します。')
else:
    # マージ: GitHub にあって Drive にないレコードを追加
    _gh = sqlite3.connect(GH_DB)
    _gh_races = _gh.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
    _gh_max = _gh.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
    print(f'GitHub: {_gh_races}レース / 最新{_gh_max}')

    # Drive に ATTACH して差分 INSERT
    _drv = sqlite3.connect(DRIVE_DB)
    _drv.execute(f"ATTACH DATABASE '{GH_DB}' AS gh")

    # race_history マージ
    _drv.execute("""
        INSERT OR IGNORE INTO race_history
        SELECT * FROM gh.race_history
        WHERE race_id NOT IN (SELECT race_id FROM race_history)
    """)
    r_added = _drv.execute("SELECT changes()").fetchone()[0]

    # horse_history マージ
    _drv.execute("""
        INSERT OR IGNORE INTO horse_history
        SELECT * FROM gh.horse_history
        WHERE race_id NOT IN (SELECT DISTINCT race_id FROM horse_history)
    """)
    h_added = _drv.execute("SELECT changes()").fetchone()[0]

    _drv.commit()
    _drv.execute("DETACH DATABASE gh")

    # 結果確認
    _new_races = _drv.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
    _new_horses = _drv.execute("SELECT COUNT(*) FROM horse_history").fetchone()[0]
    _new_max = _drv.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
    _drv.close()
    _gh.close()

    print(f'\nマージ結果:')
    print(f'  レース: {_drv_races} → {_new_races} (+{r_added})')
    print(f'  出走:   {_drv_horses} → {_new_horses} (+{h_added})')
    print(f'  最新:   {_drv_max} → {_new_max}')

    if r_added == 0:
        print('\n既に最新です。')
    else:
        print(f'\n✅ {r_added}レース追加完了')

# クリーンアップ
shutil.rmtree(TMP_REPO, ignore_errors=True)

In [ ]:
# == セル2: src/ 強制アップデート（GitHub最新コードを取得）==================
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())
files = [
    # tools
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/build_training_data.py',
    'src/tools/train_xgb.py',
    'src/tools/calibrate_xgb.py',
    'src/tools/generate_style_advantage.py',
    'src/tools/train_pace_model.py',
    'src/tools/shap_diagnosis.py',
    # features
    'src/features/engine.py',
    'src/features/speed_index.py',
    'src/features/horse_type.py',
    # utils
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    # scraper
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    # models
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/calibration_xgb.py',
    'src/models/predict.py',
    # betting
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
    'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py',
    'src/betting/rating_calibration.py',
    'src/betting/payout_estimator.py',
    'src/betting/dual_model.py',
    'src/betting/bet_optimizer.py',
    'src/betting/shadow.py',
]
_failed = []
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for _retry in range(3):
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
            print(f'OK {rel}')
            break
        except Exception as _e:
            if _retry < 2:
                _time.sleep(2 ** _retry)
            else:
                print(f'FAIL {rel}: {_e}')
                _failed.append(rel)

# data/*.json も最新版に更新
for rel_data in ['data/course_profiles.json', 'data/note_schema.json']:
    dest = f'{BASE_DIR}/{rel_data}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{rel_data}?nocache={_cb}', dest)
        print(f'OK {rel_data}')
    except Exception as e:
        print(f'SKIP {rel_data}: {e}')

if _failed:
    print(f'\n!! {len(_failed)}件失敗: {_failed}')
    print('   -> Colabランタイムを再起動してセル2を再実行してください')
else:
    print()

# モジュールキャッシュクリア
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# 主要機能の存在確認
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_speed_fig_last', 'f_popularity', 'f_style_course_fit',
            'predict_race_pace', '_build_pace_features_for_inference',
            '_JOCKEY_PACE_STATS', '_XGB_RESIDUAL']:
    _ok = _kw in _eng_src
    print(f'  {"OK" if _ok else "NG"} engine.py: {_kw}')

with open(f'{BASE_DIR}/src/tools/train_pace_model.py') as _f:
    _pm_src = _f.read()
for _kw in ['_build_pace_percentiles', '_PACE_PERCENTILE_CACHE', '_dist_zone']:
    _ok = _kw in _pm_src
    print(f'  {"OK" if _ok else "NG"} train_pace_model.py: {_kw}')

# horse_type.py の存在確認
with open(f'{BASE_DIR}/src/features/horse_type.py') as _f:
    _ht_src = _f.read()
for _kw in ['calc_distance_features', 'calc_agari_ability', 'calc_optimal_distance']:
    _ok = _kw in _ht_src
    print(f'  {"OK" if _ok else "NG"} horse_type.py: {_kw}')

print('\ndone')

In [ ]:
# == セル3: 展開予測モデル再学習（19特徴量ペースモデル）====================
# ペース分類をパーセンタイルベースに改善 + 19特徴量で学習
# -> data/pace_model.pkl + data/jockey_pace_stats.json が生成される
# -> 次のセルでbuild_training_dataが参照し、XGBの特徴量にも反映
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_pace_model import train_pace_model

pace_result = train_pace_model(BASE_DIR)
print(f'\n== ペースモデル結果 ==')
print(f'精度: {pace_result.get("accuracy", "N/A")}')
print(f'分布: {pace_result.get("distribution", "N/A")}')
print(f'特徴量数: {pace_result.get("n_features", "N/A")}')

# 検証: 分布が偏っていないか
dist = pace_result.get('distribution', {})
if dist:
    vals = list(dist.values())
    max_ratio = max(vals) / max(min(vals), 0.01)
    if max_ratio > 5:
        print(f'\n!! 分布が偏っています (max/min={max_ratio:.1f})。閾値を確認してください')
    else:
        print(f'\nOK 分布バランス良好 (max/min={max_ratio:.1f})')

In [ ]:
# == セル4: speed_index キャッシュ再構築 + 学習データ生成 ===================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.features.speed_index import rebuild_speed_index_cache
rebuild_speed_index_cache(BASE_DIR)

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# コース×脚質の style_advantage を history.db 実データで更新
from src.tools.generate_style_advantage import generate_style_advantage
generate_style_advantage(BASE_DIR)

# 学習データ生成（全特徴量入りCSV）
from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

# 特徴量充足率確認
print('\n== 特徴量充足率 ==')
CHECK_COLS = [
    'f_popularity', 'f_pop_last', 'f_pop_avg', 'f_beat_market_rate',
    'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max',
    'f_style_course_fit', 'f_pace_fit', 'f_style_total_fit',
    'f_finish_time_avg', 'f_time_diff_avg',
    'f_member_level_avg',
    'f_agari_ability', 'f_stamina_score', 'f_speed_score',
    'f_optimal_distance', 'f_dist_vs_optimal', 'f_dist_confidence',
]
for c in CHECK_COLS:
    if c in df_all.columns:
        pct = 100 * df_all[c].notna().sum() / len(df_all)
        mark = 'OK' if pct > 50 else 'LOW'
        print(f'  {mark} {c}: {pct:.1f}%')
    else:
        print(f'  NG {c}: 列なし')
print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')

In [ ]:
# == セル5: XGB再学習（通常モデル） + キャリブレーション ====================
# 日付はCSVの実データ範囲から自動算出（直近4週間=val、それ以前=train）
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

import pandas as _pd
from datetime import datetime as _dt, timedelta as _td

# CSVの実データから日付範囲を検出
_csv_path = f'{BASE_DIR}/data/horse_features.csv'
_df_dates = _pd.read_csv(_csv_path, usecols=['date'])
_df_dates['date_obj'] = _pd.to_datetime(
    _df_dates['date'].astype(str).str.replace('-', '', regex=False).str[:8],
    format='%Y%m%d', errors='coerce'
)
_df_dates = _df_dates.dropna(subset=['date_obj'])
_min_date = _df_dates['date_obj'].min().strftime('%Y-%m-%d')
_max_date = _df_dates['date_obj'].max().strftime('%Y-%m-%d')
print(f'CSVデータ範囲: {_min_date} 〜 {_max_date}')

# val期間を4週間（28日）に拡大（2週間だとAUC分散が大きすぎる）
_max_dt = _dt.strptime(_max_date, '%Y-%m-%d')
_val_start_dt = _max_dt - _td(days=27)
_train_end = (_val_start_dt - _td(days=1)).strftime('%Y-%m-%d')
_val_start = _val_start_dt.strftime('%Y-%m-%d')
_val_end = _max_date
print(f'Train: 〜{_train_end}  Val: {_val_start}〜{_val_end} (4週間)')

from src.tools.train_xgb import train_xgb
from src.tools.calibrate_xgb import run_xgb_calibration

metrics = train_xgb(BASE_DIR,
    train_end=_train_end,
    val_start=_val_start,
    val_end=_val_end)

print(f'\n== 再学習結果 ==')
print(f'AUC: {metrics.get("auc", "N/A")}')
print(f'旧AUC: {metrics.get("old_auc", "N/A")}')
print(f'特徴量数: {metrics.get("n_features", "N/A")}')

auc = metrics.get('auc', 0)
if auc >= 0.75:
    run_xgb_calibration(BASE_DIR)
    print('OK キャリブレーション完了')
else:
    print(f'!! AUC={auc:.4f} < 0.75 -> キャリブレーションスキップ')

if auc >= 0.82:
    print(f'\n>>> AUC {auc:.4f} >= 0.82: 現行水準維持。セル8でpush可能')
elif auc >= 0.80:
    print(f'\n>>> AUC {auc:.4f}: やや低下。feature_importanceを確認してからpush判断')
else:
    print(f'\n>>> AUC {auc:.4f} < 0.80: 問題あり。pushしないこと')

In [ ]:
# == セル6: 残差学習モデル（市場コピー排除実験）============================
# f_popularity を特徴量から除外し、logit(p_market) を base_margin として渡す。
# モデルは「市場からのズレ」だけを学習 -> AI独自の予測力を測定できる。
# 結果は別ファイルに保存 -> 現行モデルには影響しない。
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb

res_metrics = train_xgb(BASE_DIR,
    train_end=_train_end,
    val_start=_val_start,
    val_end=_val_end,
    residual=True)

print(f'\n== 残差学習結果 ==')
print(f'残差AUC: {res_metrics.get("auc", "N/A")}')
print(f'通常AUC: {metrics.get("auc", "N/A")}  (セル5の結果)')

res_auc = res_metrics.get('auc', 0)
normal_auc = metrics.get('auc', 0)
if res_auc >= normal_auc * 0.95:
    print(f'\n>>> 残差モデルが通常の95%以上: AI独自の予測力あり')
    print('    本番切替する場合はセル9のFILESに_residualファイルを追加')
else:
    gap = (1 - res_auc / max(normal_auc, 0.01)) * 100
    print(f'\n>>> 残差モデルが通常より{gap:.1f}%低い: 予測力の大半は市場コピー')
    print('    現行モデルを維持')

In [ ]:
# == セル7: 統合テスト =====================================================
import pickle, json, numpy as np, pandas as pd

with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f: model = pickle.load(f)
with open(f'{BASE_DIR}/data/xgb_feature_cols.json')        as f: meta  = json.load(f)
cols = meta['feature_cols']
with open(f'{BASE_DIR}/data/xgb_calibrator.pkl', 'rb')     as f: calib = pickle.load(f)

print(f'モデル特徴量数: {len(cols)}')
print(f'学習日時: {meta.get("trained_at", "不明")}')
print(f'Val AUC: {meta.get("val_auc", "不明")}')
print(f'キャリブレーター: {type(calib).__name__}')

# 実データで cal_prob 合計確認（valの各レース）
_csv = pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
_csv['date_obj'] = pd.to_datetime(
    _csv['date'].astype(str).str.replace('-', '', regex=False).str[:8],
    format='%Y%m%d', errors='coerce')
_csv = _csv.dropna(subset=['date_obj'])
_val_data = _csv[_csv['date_obj'] >= pd.Timestamp(_val_start)].copy()

if len(_val_data) > 0:
    _sums = []
    for rid, grp in _val_data.groupby('race_id'):
        X = grp[cols].reindex(columns=cols, fill_value=5.0).fillna(5.0)
        raw = model.predict_proba(X)[:, 1]
        cal = np.array(calib.transform(raw))
        _sums.append(cal.sum())
    _sums = np.array(_sums)
    print(f'\n== Val実データ cal_prob合計（理論≒3.0）==')
    print(f'  レース数: {len(_sums)}')
    print(f'  平均: {_sums.mean():.3f}')
    print(f'  中央値: {np.median(_sums):.3f}')
    print(f'  範囲: [{_sums.min():.3f}, {_sums.max():.3f}]')
    cal_ok = 2.5 <= _sums.mean() <= 3.5
    print(f'  合計チェック: {"OK" if cal_ok else "NG（キャリブレーション要確認）"}')

# 特徴量重要度トップ20
HIGHLIGHT = {
    'f_popularity': 'MARKET', 'f_pop_last': 'MARKET', 'f_pop_avg': 'MARKET',
    'f_beat_market_rate': 'MARKET',
    'f_speed_fig_last': 'SPEED', 'f_speed_fig_avg': 'SPEED', 'f_speed_fig_max': 'SPEED',
    'f_pace': 'PACE',
    'f_agari_ability': 'DIST_TYPE', 'f_stamina_score': 'DIST_TYPE',
    'f_speed_score': 'DIST_TYPE', 'f_optimal_distance': 'DIST_TYPE',
    'f_dist_vs_optimal': 'DIST_TYPE', 'f_dist_confidence': 'DIST_TYPE',
    'f_speed_x_shortening': 'DIST_TYPE', 'f_stamina_x_extension': 'DIST_TYPE',
}
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n== 特徴量重要度 Top 20 ==')
for name, imp in imps:
    tag = HIGHLIGHT.get(name, '')
    print(f'  {name:<42} {imp*100:5.2f}%  {tag}')

# カテゴリ別の合計重要度
categories = {
    'MARKET': ['f_popularity', 'f_pop_last', 'f_pop_avg', 'f_beat_market_rate'],
    'SPEED': ['f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max', 'f_speed_fig_trend'],
    'DIST_TYPE': ['f_agari_ability', 'f_stamina_score', 'f_speed_score',
                  'f_optimal_distance', 'f_dist_vs_optimal', 'f_dist_change',
                  'f_speed_x_shortening', 'f_stamina_x_extension', 'f_dist_confidence'],
}
print('\n== カテゴリ別重要度 ==')
for cat, keys in categories.items():
    cat_imp = sum(imp for n, imp in zip(cols, model.feature_importances_) if n in keys)
    print(f'  {cat:<12}: {cat_imp*100:.1f}%')

# 残差モデルがあれば比較表示
import os
res_path = f'{BASE_DIR}/data/xgb_feature_cols_residual.json'
if os.path.exists(res_path):
    with open(res_path) as f:
        res_meta = json.load(f)
    print(f'\n== 残差モデル情報 ==')
    print(f'  残差AUC: {res_meta.get("val_auc", "不明")}')
    print(f'  特徴量数: {len(res_meta.get("feature_cols", []))}')
    print(f'  residualフラグ: {res_meta.get("residual", False)}')

In [ ]:
# == セル8: 再学習モデルを GitHub main にプッシュ ===========================
# これを実行しないと、Driveで再学習したモデルが本番(GitHub Actions予想)に反映されない。
# 前提: Colabシークレットに GITHUB_PAT を登録し、ノートの「ノートブックからの
#       アクセス」をONにしておくこと。
import requests, base64, json as _json, os as _os
from google.colab import userdata

GITHUB_PAT = userdata.get('GITHUB_PAT')
REPO  = 'hanagenuku/keiba_ai'

# pushするファイル一覧（存在するもののみ）
FILES = [
    'data/xgb_fukusho_model.pkl',
    'data/xgb_feature_cols.json',
    'data/xgb_calibrator.pkl',
    'data/pace_model.pkl',
    'data/jockey_pace_stats.json',
    'data/speed_index_cache.pkl',
    # 残差学習モデル（セル6で生成）
    'data/xgb_fukusho_model_residual.pkl',
    'data/xgb_feature_cols_residual.json',
]

def push_file(relpath, pat, message):
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=h, params={'ref': 'main'})
    sha = r.json().get('sha') if r.status_code == 200 else None
    with open(f'{BASE_DIR}/{relpath}', 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    payload = {'message': message, 'content': content, 'branch': 'main'}
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=h, json=payload)
    ok = r.status_code in (200, 201)
    status = 'OK' if ok else f'NG({r.status_code})'
    print(f'{status} {relpath}')
    if not ok:
        print(f'  -> {r.json().get("message", "")}')
    return ok

# メタ情報の確認（セル5で新モデルが採用されたか）
with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    _meta = _json.load(f)
n = len(_meta['feature_cols'])
auc_val = _meta.get('val_auc', 0)
trained_at = _meta.get('trained_at', '不明')
auc_str = f'{auc_val:.4f}'

print(f'=== push前確認 ===')
print(f'  特徴量数: {n}')
print(f'  Val AUC : {auc_str}')
print(f'  学習日時: {trained_at}')

# セル5の結果と一致するか確認
_current_auc = metrics.get('auc', 0) if 'metrics' in dir() else 0
if _current_auc > 0 and abs(auc_val - _current_auc) > 0.001:
    print(f'\n⚠ セル5のAUC({_current_auc:.4f})とファイルのAUC({auc_val:.4f})が不一致。')
    print('  旧モデルがpushされる可能性があります。')
    print('  続行するには下のコメントアウトを外してください。')
    raise RuntimeError('AUC不一致: セル5で新モデルが不採用だった可能性。確認してからpush')

msg = f'model: retrain v4 {n}feat AUC={auc_str}'

# 残差モデル情報
res_path = f'{BASE_DIR}/data/xgb_feature_cols_residual.json'
if _os.path.exists(res_path):
    with open(res_path) as f:
        _res = _json.load(f)
    print(f'残差AUC: {_res.get("val_auc", "N/A")}')
    msg += f' residual_AUC={_res.get("val_auc", 0):.4f}'

print(f'\nコミットメッセージ: {msg}\n')

pushed = 0
for rel in FILES:
    if not _os.path.exists(f'{BASE_DIR}/{rel}'):
        print(f'SKIP（未生成）: {rel}')
        continue
    if push_file(rel, GITHUB_PAT, msg):
        pushed += 1

print(f'\n完了: {pushed}/{len(FILES)} ファイルをpush')